In [2]:
import pandas as pd
import numpy as np
import sqlite3

In [20]:
with sqlite3.connect("player_stats.db") as conn:
      df = pd.read_sql_query("SELECT * FROM tm_injuries", conn)
df

,id,tm_id,injury_type,date_from,date_to,matches_missed,scraped_at
0,1,554389,Rest,25/02/2025,27/02/2025,1.0,2026-05-30 20:21:56
1,2,554389,Shoulder injury,18/10/2022,25/10/2022,1.0,2026-05-30 20:21:56
2,3,554389,Quarantine,13/10/2021,19/10/2021,1.0,2026-05-30 20:21:56
3,4,554389,Quarantine,22/08/2020,01/09/2020,NaN,2026-05-30 20:21:56
4,5,88348,Calf injury,20/11/2025,30/01/2026,15.0,2026-05-30 20:22:02
...,...,...,...,...,...,...,...
1829,1830,459115,Hamstring injury,25/08/2024,20/09/2024,4.0,2026-05-30 21:12:43
1830,1831,459115,Inner ligament stretch of the knee,03/11/2022,31/12/2022,5.0,2026-05-30 21:12:43
1831,1832,459115,unknown injury,05/09/2022,29/10/2022,11.0,2026-05-30 21:12:43
1832,1833,459115,Quarantine,11/08/2020,25/08/2020,4.0,2026-05-30 21:12:43


In [33]:
df["date_to"].dtype
#df["date_to"].head(10)
#df["date_to"].isna().sum() ~ 0!!

df["date_to_parsed"] = pd.to_datetime(df["date_to"])
#df["date_to_parsed"]

df_events_end_sorted_ascending = df.sort_values(by = "date_to_parsed",ascending=True)
df_events_end_sorted_descending = df.sort_values(by = "date_to_parsed",ascending=False)
df_events_end_sorted_descending

C:\Users\skous\AppData\Local\Temp\ipykernel_92692\2080869454.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date_to_parsed"] = pd.to_datetime(df["date_to"])


,id,tm_id,injury_type,date_from,date_to,matches_missed,scraped_at,date_to_parsed
88,89,386726,Cruciate ligament tear,07/05/2026,31/12/2026,3.0,2026-05-30 20:24:10,2026-12-31
1556,1557,1059172,Cruciate ligament tear,22/03/2026,21/09/2026,10.0,2026-05-30 21:01:40,2026-09-21
1113,1114,357161,Cruciate ligament tear,04/12/2025,08/06/2026,25.0,2026-05-30 20:45:17,2026-06-08
1062,1063,615300,Groin injury,21/11/2025,18/05/2026,27.0,2026-05-30 20:43:31,2026-05-18
570,571,905184,Groin problems,16/05/2026,18/05/2026,1.0,2026-05-30 20:31:58,2026-05-18
...,...,...,...,...,...,...,...,...
1742,1743,591360,unknown injury,28/02/2026,NaN,10.0,2026-05-30 21:08:57,NaT
1747,1748,242758,unknown injury,02/05/2026,NaN,4.0,2026-05-30 21:09:11,NaT
1753,1754,290261,unknown injury,21/03/2026,NaN,7.0,2026-05-30 21:09:24,NaT
1781,1782,735366,unknown injury,03/04/2026,NaN,6.0,2026-05-30 21:09:54,NaT


In [35]:
df["date_to_parsed"].isna().sum()

np.int64(33)

In [37]:
counts = {}
validDf = df[df["date_to_parsed"].notna()]

for i in range(2010,2027):
    counts[i] = 0

for date in validDf["date_to_parsed"] :
    year = str(date).split("-")[0]
    counts[int(year)] += 1

counts

{2010: 1,
 2011: 4,
 2012: 2,
 2013: 10,
 2014: 11,
 2015: 28,
 2016: 45,
 2017: 72,
 2018: 51,
 2019: 80,
 2020: 107,
 2021: 157,
 2022: 173,
 2023: 225,
 2024: 309,
 2025: 314,
 2026: 212}

In [39]:
#keeping only the dates we will build our initial formula on

mask = (
      (df["date_to_parsed"] >= '2022-08-23') &
      (df["date_to_parsed"] <= '2025-08-22')
  )
df_sample = df[mask]


In [42]:
xg = pd.read_sql("SELECT match_id, player_team, sot FROM sofascore_xg", conn)

# team_for = (xg.groupby(['match_id','player_team'], as_index=False)['shots']
#               .sum().rename(columns={'player_team':'team', 'shots':'shots_for'}))
#
# pair = team_for.merge(team_for, on='match_id', suffixes=('','_opp'))
# pair = pair[pair['team'] != pair['team_opp']]
#
# pair['shots_against'] = pair['shots_for_opp']
# team_match = pair[['match_id','team','team_opp','shots_for','shots_against']]
#
# team_match

xg

,match_id,player_team,sot
0,15866537,MGS Panserraikos,1
1,15866537,MGS Panserraikos,0
2,15866537,MGS Panserraikos,0
3,15866537,AEL Novibet,2
4,15866537,MGS Panserraikos,1
...,...,...,...
3245,14151968,APO Levadiakos,1
3246,14151968,APO Levadiakos,2
3247,14151968,Olympiacos FC,0
3248,14151968,Olympiacos FC,0
